In [40]:
import re
import random
import os
from collections import Counter

import numpy as np
import pandas as pd
import datasets
import spacy
from spacy.tokens import DocBin
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login()
os.makedirs('../data_spacy', exist_ok=True)

In [41]:
ds = datasets.load_dataset('scanpatch/pii-ner-corpus-synthetic-controlled')
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'entity_starts', 'entity_ends', 'entity_labels', 'entity_texts', 'source', 'noisy'],
        num_rows: 4786
    })
    test: Dataset({
        features: ['text', 'entity_starts', 'entity_ends', 'entity_labels', 'entity_texts', 'source', 'noisy'],
        num_rows: 532
    })
})


In [42]:
_UKRAINIAN = re.compile(r'[\u0407\u0457\u0404\u0454\u0406\u0456]')
_CYRILLIC  = re.compile(r'[\u0430-\u044f\u0410-\u042f\u0451\u0401]')
_ALPHA     = re.compile(r'[a-zA-Z\u0430-\u044f\u0410-\u042f\u0451\u0401]')

def is_russian_cyrillic(text: str, min_ratio: float = 0.7) -> bool:
    alpha = len(_ALPHA.findall(text))
    if alpha == 0:
        return False
    if len(_CYRILLIC.findall(text)) / alpha < min_ratio:
        return False
    if _UKRAINIAN.search(text):
        return False
    return True

sp_train = ds['train'].filter(lambda x: is_russian_cyrillic(x['text']))
sp_test  = ds['test'].filter(lambda x: is_russian_cyrillic(x['text']))
print(f"scanpatch train: {len(ds['train'])} → {len(sp_train)}")
print(f"scanpatch test:  {len(ds['test'])} → {len(sp_test)}")

scanpatch train: 4786 → 2238
scanpatch test:  532 → 233


In [43]:
all_sp_labels = [lbl for row in sp_train for lbl in row['entity_labels']]
print("Метки в scanpatch train:")
for label, count in Counter(all_sp_labels).most_common():
    print(f"  {label!r}: {count}")

Метки в scanpatch train:
  'name': 1902
  'first_name': 1875
  'ip': 1626
  'last_name': 1404
  'date': 1059
  'address_city': 771
  'middle_name': 724
  'mobile_phone': 609
  'address': 599
  'address_street': 527
  'document_number': 494
  'organization': 422
  'email': 321
  'address_house': 263
  'name_initials': 169
  'snils': 167
  'address_geolocation': 149
  'military_individual_number': 117
  'vehicle_number': 114
  'address_district': 74
  'tin': 68
  'address_region': 52
  'nickname': 47
  'address_apartment': 27
  'address_country': 18
  'address_building': 10
  'address_postal_code': 7


In [44]:
NAME_LABELS_SP = {'name', 'first_name', 'last_name', 'middle_name', 'name_initials'}
ADDRESS_LABELS_SP = {
    'address', 'address_city', 'address_street', 'address_house',
    'address_building', 'address_apartment', 'address_district',
    'address_region', 'address_country', 'address_postal_code', 'address_geolocation',
}
KEEP = {'NAME', 'ADDRESS'}

def normalize_sp_labels(labels):
    result = []
    for l in labels:
        ll = l.lower()
        if ll in NAME_LABELS_SP:
            result.append('NAME')
        elif ll in ADDRESS_LABELS_SP:
            result.append('ADDRESS')
        else:
            result.append(l)
    return result

In [45]:
rows = []
for row in sp_train:
    labels_norm = normalize_sp_labels(row['entity_labels'])
    rows.append({
        'text': row['text'],
        'old_labels': row['entity_labels'],
        'new_labels': labels_norm,
    })
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', None)
pd.DataFrame(rows)

,text,old_labels,new_labels
0,"Виктор пролистал документы и задержал взгляд на протоколе обхода, составленном старшим инженером Игорем Нестеровым. Тот упоминал, что вскрытие архивного шкафа в серверной №4 было выполнено с использованием кодового замка, привязанного к терминалу 192.168.32.44. В журнале присутствовали следы неожиданных обращений, напоминающих переадресации, а строка ip 10.11.90.210 стояла дважды — будто кто-то пытался замаскировать повторное обращение под обычное обновление конфигурации.","[first_name, name, first_name, last_name, ip, ip]","[NAME, NAME, NAME, NAME, ip, ip]"
1,Раздел V. Процедуры Обслуживания и Контроль Доступа,[],[]
2,"Порт создаёт новую реальность каждый раз, когда на горизонте появляется корабль, идущий к берегу, где на бетонных плитах мокнут следы от колес грузовиков. Южный порт Одессы, район Пересыпь, складский терминал возле улицы Молитовской кажется живым организмом, который дышит морским ветром. В дальнем углу, рядом с ангаром №12, виднеется табличка «Компания “Морские линии”, Одесса», а на дверях плохо читаемый номер телефона +38(093)745-88-01, написанный маркером поверх старых цифр. Здесь всегда пахнет солью, дизелем и ржавчиной, а люди говорят короткими фразами, словно экономят слова. Дальше, ближе к причалу №3, стоит маленькое офисное помещение, где секретарь Наталья отвечает на письма, приходящие на адрес info.morline@ukrmail.net, хотя некоторые пишут по ошибке на inf.morline@ukrmaill.net, и тогда письмо теряется в глубинах интернета.","[address, address_city, address_district, address_street, address_city, mobile_phone, first_name, email]","[ADDRESS, ADDRESS, ADDRESS, ADDRESS, ADDRESS, mobile_phone, NAME, email]"
3,"В протоколе № 411 от 17.10.2033 указано: «Ожидается усиление». Частный исследователь из Одессы (Французский бульвар) зафиксировал 06.09.2040, подпись дрожит. Есть документ с датой 2100-01-01 и запись: «Начало нулевого цикла». Другой документ: 00-00-3000 — будто речь о будущем, где нет привычного календаря.","[document_number, date, address, address_city, address_street, date, date, date]","[document_number, date, ADDRESS, ADDRESS, ADDRESS, date, date, date]"
4,"И когда туман наконец рассеивается, остаётся ощущение непрерывного движения и множества следов, где корректные и некорректные записи существуют на равных правах, переплетаясь с дорогами, машинами и точками на карте, не требуя объяснений и не превращаясь в отчёт, а оставаясь просто частью странного, текучего рассказа.",[],[]
...,...,...,...
2233,"Дубль мерцает. Именно дубль реальности. Когда-то один прохожий видел на дверях служебной комнаты табличку: «Действительно: до 2045/11/01», а через секунду на том же месте было «01-11-45». Он моргнул — и надпись исчезла. На камерах наблюдения отметки времени тоже пляшут: на кадре охранник открывает дверь в 2025-07-01 09:61, хотя минут шестьдесят первой быть не должно.","[date, date, date]","[date, date, date]"
2234,"Внутри дом оказался пустым, но на столе лежала стопка писем. В верхнем из них стояла надпись: «Для Андрея Львовича Мироненко», но почерк указывал, что автор, возможно, сначала написал другое — под светом окна Андрею показалось, что под этой надписью угадывается стёртая фраза «Для Мироненко Андрей Львович». В письме говорилось о человеке по имени Степан Николаевич Руденко, который якобы хранил у себя часть записей Марины Александровны. И снова Андрей сталкивался с таинственной текучестью имён: в письмах этот человек ухудшался и исправлялся в бесконечных вариациях — Руденко Степан Николаевич, Николаевич Степан Руденко, Степан Руденко Николаевич, каждое из которых казалось отражением другого.","[name, first_name, middle_name, middle_name, last_name, first_name, name, first_name, name, first_name, middle_name, last_name, name, first_name, middle_name, name, name, name]","[NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME, NAME]"
2235,"В Архангельск он прибыл рано утром 3 декабря. Город встретил его холодом, запахом 

In [46]:
path_ds = '../Detailed-NER-Dataset-RU/dataset/detailed-ner_dataset-ru.pickle'
alex_df = pd.read_pickle(path_ds)
print(alex_df.info())
print("\nВсе теги:")
print(alex_df['ner_tags'].explode().value_counts().to_string())

<class 'pandas.DataFrame'>
RangeIndex: 7532 entries, 0 to 7531
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   tokens    7532 non-null   object
 1   ner_tags  7532 non-null   object
dtypes: object(2)
memory usage: 117.8+ KB
None

Все теги:
ner_tags
O                47908
U-LAST_NAME       1071
U-FIRST_NAME       912
U-COUNTRY          714
U-CITY             648
U-REGION           370
U-MIDDLE_NAME      309
I-HOUSE            126
U-DISTRICT         104
U-STREET            98
B-HOUSE             94
L-HOUSE             92
B-COUNTRY           90
L-COUNTRY           89
B-STREET            37
L-STREET            36
B-CITY              29
L-CITY              28
U-HOUSE             26
I-COUNTRY           18
I-STREET            16
I-CITY              13
B-LAST_NAME         13
L-LAST_NAME         12
B-REGION            11
L-REGION            11
I-LAST_NAME          6
B-FIRST_NAME         6
B-DISTRICT           6
L-DISTRICT         

In [47]:
# AlexKly: BIOLU-схема (B/I/O/L/U)
# Сохраняем оригинальный лейбл — нужен для мержа ФИО-компонентов и адресных типов
NAME_TAGS_AL    = {'LAST_NAME', 'FIRST_NAME', 'MIDDLE_NAME'}
ADDRESS_TAGS_AL = {'COUNTRY', 'REGION', 'CITY', 'DISTRICT', 'STREET', 'HOUSE'}

def normalize_bio_tags(tags):
    result = []
    for tag in tags:
        if tag == 'O' or '-' not in tag:
            result.append('O')
            continue
        prefix, label = tag.split('-', 1)
        if label in NAME_TAGS_AL or label in ADDRESS_TAGS_AL:
            result.append(tag)  # сохраняем оригинальный тег: U-FIRST_NAME, B-CITY и т.д.
        else:
            result.append('O')
    return result

alex_df['ner_tags_norm'] = alex_df['ner_tags'].apply(normalize_bio_tags)

norm_counts = Counter(tag for tags in alex_df['ner_tags_norm'] for tag in tags)
print("Теги после нормализации:")
for tag, count in norm_counts.most_common():
    if tag != 'O':
        print(f"  {tag!r}: {count}")

Теги после нормализации:
  'U-LAST_NAME': 1071
  'U-FIRST_NAME': 912
  'U-COUNTRY': 714
  'U-CITY': 648
  'U-REGION': 370
  'U-MIDDLE_NAME': 309
  'I-HOUSE': 126
  'U-DISTRICT': 104
  'U-STREET': 98
  'B-HOUSE': 94
  'L-HOUSE': 92
  'B-COUNTRY': 90
  'L-COUNTRY': 89
  'B-STREET': 37
  'L-STREET': 36
  'B-CITY': 29
  'L-CITY': 28
  'U-HOUSE': 26
  'I-COUNTRY': 18
  'I-STREET': 16
  'I-CITY': 13
  'B-LAST_NAME': 13
  'L-LAST_NAME': 12
  'B-REGION': 11
  'L-REGION': 11
  'I-LAST_NAME': 6
  'B-FIRST_NAME': 6
  'B-DISTRICT': 6
  'L-DISTRICT': 6
  'L-FIRST_NAME': 5
  'I-REGION': 4
  'B-MIDDLE_NAME': 2
  'L-MIDDLE_NAME': 1


In [48]:
def extract_spans_biolu(tokens, tags):
    spans, cur_tokens, cur_label = [], [], None
    for token, tag in zip(tokens, tags):
        if tag == 'O' or '-' not in tag:
            if cur_label:
                spans.append((' '.join(cur_tokens), cur_label))
                cur_tokens, cur_label = [], None
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'U':
            if cur_label:
                spans.append((' '.join(cur_tokens), cur_label))
            spans.append((token, label))
            cur_tokens, cur_label = [], None
        elif prefix == 'B':
            if cur_label:
                spans.append((' '.join(cur_tokens), cur_label))
            cur_tokens, cur_label = [token], label
        elif prefix in ('I', 'L') and cur_label:
            cur_tokens.append(token)
            if prefix == 'L':
                spans.append((' '.join(cur_tokens), cur_label))
                cur_tokens, cur_label = [], None
        else:
            if cur_label:
                spans.append((' '.join(cur_tokens), cur_label))
            cur_tokens, cur_label = [], None
    if cur_label:
        spans.append((' '.join(cur_tokens), cur_label))
    return spans

rows = []
for _, row in alex_df.iterrows():
    tokens = [t for t in row['tokens'] if t]
    old_spans = extract_spans_biolu(tokens, row['ner_tags'])
    new_spans = extract_spans_biolu(tokens, row['ner_tags_norm'])
    rows.append({
        'text': ' '.join(tokens),
        'old_labels': [f"{lbl}:{txt}" for txt, lbl in old_spans],
        'new_labels': [f"{lbl}:{txt}" for txt, lbl in new_spans],
    })
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', None)
pd.DataFrame(rows)

,text,old_labels,new_labels
0,dnsmasq 3753720 1 0 Mar10 ? 00:00:02 /usr/sbin/dnsmasq -k,[],[]
1,2022-09-09 12:37:10,[],[]
2,Повар судовой,[],[]
3,профилирование: SafeData,[],[]
4,"Кораблестроение, океанотехника и системотехника объектов морской инфраструктуры",[],[]
...,...,...,...
7527,CHANGE_PIN,[],[]
7528,Водоотведение,[],[]
7529,DisplayPromotionalOffersForProducts,[],[]
7530,Союзнефтегаз,[],[]


In [49]:
nlp = spacy.blank("ru")

ADDR_SEPARATORS = set(' ,.:;/-()№#\n\t')
NAME_SEPARATORS = {' ', '-'}

ADDR_PREFIXES = {
    'г', 'гор', 'ул', 'пр', 'пр-т', 'пер', 'пл', 'б-р', 'бул',
    'д', 'дом', 'кв', 'стр', 'корп', 'обл', 'р-н', 'пос', 'с', 'дер', 'мкр', 'наб'
}

_ADDR_MERGE_ALLOWED = {
    frozenset({'COUNTRY', 'REGION'}),
    frozenset({'COUNTRY', 'CITY'}),
    frozenset({'REGION', 'CITY'}),
    frozenset({'REGION', 'DISTRICT'}),
    frozenset({'CITY', 'DISTRICT'}),
    frozenset({'CITY', 'STREET'}),
    frozenset({'CITY', 'HOUSE'}),
    frozenset({'DISTRICT', 'STREET'}),
    frozenset({'DISTRICT', 'HOUSE'}),
    frozenset({'STREET', 'HOUSE'}),
    frozenset({'STREET', 'STREET'}),
    frozenset({'HOUSE', 'HOUSE'}),
}

SP_ADDR_TYPE_MAP = {
    'address':             'ADDRESS',
    'address_city':        'CITY',
    'address_street':      'STREET',
    'address_house':       'HOUSE',
    'address_building':    'HOUSE',
    'address_apartment':   'HOUSE',
    'address_district':    'DISTRICT',
    'address_region':      'REGION',
    'address_country':     'COUNTRY',
    'address_postal_code': 'POSTAL',
    'address_geolocation': 'GEO',
}


def _types_can_merge(type_a, type_b):
    # Эти проверки ДОЛЖНЫ быть до wildcard-правила
    # После HOUSE может идти только ещё один HOUSE — никаких улиц и новых адресов
    if type_a == 'HOUSE' and type_b != 'HOUSE':
        return False
    # Два "полных" адреса подряд = список, не один адрес
    if type_a == 'ADDRESS' and type_b == 'ADDRESS':
        return False
    # Bare ADDRESS/POSTAL/GEO мержатся с любым компонентом
    if type_a in ('ADDRESS', 'POSTAL', 'GEO') or type_b in ('ADDRESS', 'POSTAL', 'GEO'):
        return True
    if type_a == type_b and type_a in ('COUNTRY', 'CITY', 'REGION', 'DISTRICT'):
        return False
    return frozenset({type_a, type_b}) in _ADDR_MERGE_ALLOWED


def is_cyrillic_entity(text: str, min_ratio: float = 0.5) -> bool:
    cyrillic = _CYRILLIC.findall(text)
    if not cyrillic:
        return False
    alpha = len(_ALPHA.findall(text))
    return len(cyrillic) / alpha >= min_ratio


# ── SCANPATCH ────────────────────────────────────────────────────────────────

def make_char_mask(text_len, starts, ends, labels, label_set):
    mask = np.zeros(text_len, dtype=bool)
    for s, e, l in zip(starts, ends, labels):
        if l in label_set:
            mask[s:e] = True
    return mask


def mask_to_spans(mask, label):
    spans, in_span, start = [], False, 0
    for i, v in enumerate(mask):
        if v and not in_span:
            start, in_span = i, True
        elif not v and in_span:
            spans.append((start, i, label))
            in_span = False
    if in_span:
        spans.append((start, len(mask), label))
    return spans


def _char_gap_is_separator(text, start, end):
    gap = text[start:end]
    if all(c in ADDR_SEPARATORS for c in gap):
        return True
    words = [w for w in re.split(r'[ ,.:;/\-()№#]+', gap) if w]
    return all(w.lower().rstrip('.,') in ADDR_PREFIXES for w in words)


def merge_sp_addr_spans(spans, text):
    if not spans:
        return []
    spans = sorted(spans, key=lambda x: (x[0], -(x[1] - x[0])))

    cur_s, cur_e, cur_orig = spans[0]
    cur_type = SP_ADDR_TYPE_MAP.get(cur_orig, 'ADDRESS')
    result = []

    for nxt_s, nxt_e, nxt_orig in spans[1:]:
        nxt_type = SP_ADDR_TYPE_MAP.get(nxt_orig, 'ADDRESS')
        if nxt_s < cur_e:
            cur_e = max(cur_e, nxt_e)
            if nxt_type in ('STREET', 'HOUSE'):
                cur_type = nxt_type
            continue
        if _char_gap_is_separator(text, cur_e, nxt_s) and _types_can_merge(cur_type, nxt_type):
            cur_e = nxt_e
            cur_type = nxt_type
        else:
            result.append((cur_s, cur_e, 'ADDRESS'))
            cur_s, cur_e, cur_type = nxt_s, nxt_e, nxt_type

    result.append((cur_s, cur_e, 'ADDRESS'))
    return result


def scanpatch_to_docbin(dataset) -> DocBin:
    db = DocBin()
    for row in dataset:
        text = row['text']
        n = len(text)
        labels_norm = normalize_sp_labels(row['entity_labels'])

        name_mask = make_char_mask(n, row['entity_starts'], row['entity_ends'], labels_norm, {'NAME'})

        addr_spans_raw = [
            (s, e, orig_l)
            for s, e, norm_l, orig_l in zip(
                row['entity_starts'], row['entity_ends'], labels_norm, row['entity_labels']
            )
            if norm_l == 'ADDRESS'
        ]
        addr_spans = merge_sp_addr_spans(addr_spans_raw, text)

        doc = nlp.make_doc(text)
        ents = []
        for s, e, label in mask_to_spans(name_mask, 'NAME') + addr_spans:
            if not is_cyrillic_entity(text[s:e]):
                continue
            span = doc.char_span(s, e, label=label, alignment_mode="expand")
            if span is not None:
                ents.append(span)
        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)
    return db


# ── ALEXKLY ──────────────────────────────────────────────────────────────────

def parse_biolu(tokens, tags, tok_starts, tok_ends):
    """Парсит BIOLU → (start_char, end_char, norm_label, orig_type, first_tok_i, last_tok_i)."""
    entities = []
    cur_start = cur_label = cur_type = cur_first = None

    def flush(end_char, last_i):
        nonlocal cur_start, cur_label, cur_type, cur_first
        if cur_label:
            entities.append((cur_start, end_char, cur_label, cur_type, cur_first, last_i))
        cur_start = cur_label = cur_type = cur_first = None

    for i, (tag, ts, te) in enumerate(zip(tags, tok_starts, tok_ends)):
        if tag == 'O' or '-' not in tag:
            flush(tok_ends[i-1] if i > 0 else 0, i-1)
            continue
        prefix, raw_type = tag.split('-', 1)
        norm = 'NAME' if raw_type in NAME_TAGS_AL else raw_type

        if prefix == 'U':
            flush(tok_ends[i-1] if i > 0 else 0, i-1)
            entities.append((ts, te, norm, raw_type, i, i))
        elif prefix == 'B':
            flush(tok_ends[i-1] if i > 0 else 0, i-1)
            cur_start, cur_label, cur_type, cur_first = ts, norm, raw_type, i
        elif prefix in ('I', 'L'):
            if cur_label is None:
                entities.append((ts, te, norm, raw_type, i, i))
            elif prefix == 'L':
                entities.append((cur_start, te, cur_label, cur_type, cur_first, i))
                cur_start = cur_label = cur_type = cur_first = None
        else:
            flush(tok_ends[i-1] if i > 0 else 0, i-1)

    if cur_label:
        entities.append((cur_start, tok_ends[-1], cur_label, cur_type, cur_first, len(tokens)-1))
    return entities


def merge_name_entities(name_ents, tokens):
    """Мержит соседние NAME-компоненты если они — части одного ФИО."""
    if not name_ents:
        return []
    merged = []
    cur_s, cur_e, _, cur_otype, cur_fi, cur_li = name_ents[0]
    cur_types = {cur_otype}

    for ent in name_ents[1:]:
        nxt_s, nxt_e, _, nxt_otype, nxt_fi, nxt_li = ent
        between = tokens[cur_li+1 : nxt_fi]
        between_ok = all(all(c in NAME_SEPARATORS for c in tok) for tok in between)
        if between_ok and nxt_otype not in cur_types:
            cur_e, cur_li = nxt_e, nxt_li
            cur_types.add(nxt_otype)
        else:
            merged.append((cur_s, cur_e, 'NAME'))
            cur_s, cur_e, cur_otype, cur_fi, cur_li = nxt_s, nxt_e, nxt_otype, nxt_fi, nxt_li
            cur_types = {nxt_otype}

    merged.append((cur_s, cur_e, 'NAME'))
    return merged


def merge_address_entities(entities, tokens):
    """Мержит ADDRESS-компоненты и объединяет ФИО-части в единый NAME-спан."""
    name_ents = [ent for ent in entities if ent[2] == 'NAME']
    addr_ents = [ent for ent in entities if ent[2] != 'NAME']

    name_spans = merge_name_entities(name_ents, tokens)

    if not addr_ents:
        return name_spans

    merged_addr = []
    cur = addr_ents[0]

    for nxt in addr_ents[1:]:
        cur_s, cur_e, _, cur_type, cur_fi, cur_li = cur
        nxt_s, nxt_e, _, nxt_type, nxt_fi, nxt_li = nxt
        between = tokens[cur_li+1 : nxt_fi]
        between_ok = all(
            all(c in ADDR_SEPARATORS for c in tok) or
            tok.lower().rstrip('.,') in ADDR_PREFIXES
            for tok in between
        )
        if between_ok and _types_can_merge(cur_type, nxt_type):
            cur = (cur_s, nxt_e, 'ADDRESS', nxt_type, cur_fi, nxt_li)
        else:
            merged_addr.append((cur_s, cur_e, 'ADDRESS'))
            cur = nxt

    merged_addr.append((cur[0], cur[1], 'ADDRESS'))
    return name_spans + merged_addr


def alexkly_to_docbin(df) -> DocBin:
    db = DocBin()
    for _, row in df.iterrows():
        # Стрипаем токены — убираем \n и прочие пробельные символы по краям
        pairs = [(t.strip(), tg) for t, tg in zip(row['tokens'], row['ner_tags_norm']) if t and t.strip()]
        if not pairs:
            db.add(nlp.make_doc(''))
            continue
        tokens, tags = zip(*pairs)
        tokens, tags = list(tokens), list(tags)
        text = ' '.join(tokens)

        tok_starts, tok_ends = [], []
        offset = 0
        for tok in tokens:
            tok_starts.append(offset)
            tok_ends.append(offset + len(tok))
            offset += len(tok) + 1

        entities  = parse_biolu(tokens, tags, tok_starts, tok_ends)
        spans_raw = merge_address_entities(entities, tokens)

        doc = nlp.make_doc(text)
        ents = []
        for s, e, label in spans_raw:
            if not is_cyrillic_entity(text[s:e]):
                continue
            span = doc.char_span(s, e, label=label, alignment_mode="expand")
            if span is not None:
                ents.append(span)
        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)
    return db

In [50]:
# DEBUG: 30 случайных примеров — токены/теги → финальные спаны
rows_debug = []
for _, row in alex_df.sample(30, random_state=42).iterrows():
    pairs = [(t.strip(), tg) for t, tg in zip(row['tokens'], row['ner_tags_norm']) if t and t.strip()]
    if not pairs:
        continue
    tokens, tags = zip(*pairs)
    tokens, tags = list(tokens), list(tags)
    text = ' '.join(tokens)

    tok_starts, tok_ends = [], []
    offset = 0
    for tok in tokens:
        tok_starts.append(offset)
        tok_ends.append(offset + len(tok))
        offset += len(tok) + 1

    entities  = parse_biolu(tokens, tags, tok_starts, tok_ends)
    spans_raw = merge_address_entities(entities, tokens)

    rows_debug.append({
        'text': text,
        'biolu_entities': [f"{lbl}({typ}):{text[s:e]}" for s, e, lbl, typ, *_ in entities],
        'result_spans':   [f"[{lbl}] {text[s:e]!r}" for s, e, lbl in spans_raw],
    })

pd.set_option('display.max_colwidth', None)
pd.DataFrame(rows_debug)

,text,biolu_entities,result_spans
0,В Японии умер старейший мужчина на планете Титэцу Ватанабэ Весна в Дзёэцу В особом городе Дзёэцу что в японской префектуре Ниигата в воскресенье 23 февраля 2020 года умер Титэцу Ватанабэ который две недели назад 12 февраля был официально признан Книгой рекордов Гиннеса старейшим из живущих на Земле мужчин Он не дожил до 113 лет всего 11 дней,"[COUNTRY(COUNTRY):Японии, NAME(FIRST_NAME):Титэцу, NAME(LAST_NAME):Ватанабэ, CITY(CITY):Дзёэцу, CITY(CITY):Дзёэцу, CITY(CITY):Ниигата, NAME(FIRST_NAME):Титэцу, NAME(LAST_NAME):Ватанабэ, NAME(LAST_NAME):Гиннеса]","[[NAME] 'Титэцу Ватанабэ', [NAME] 'Титэцу Ватанабэ', [NAME] 'Гиннеса', [ADDRESS] 'Японии', [ADDRESS] 'Дзёэцу', [ADDRESS] 'Дзёэцу', [ADDRESS] 'Ниигата']"
1,цитрик: Citrix,[],[]
2,AGENT,[],[]
3,"kservice Кузовной сервис №1 в Москве г . Москва , ул . Дубининская , 70 стр . 1 ( ~ 14000 без стоимости бампера и без дополнительных работ )","[CITY(CITY):Москве, CITY(CITY):Москва, STREET(STREET):Дубининская, HOUSE(HOUSE):70 стр . 1]","[[ADDRESS] 'Москве', [ADDRESS] 'Москва , ул . Дубининская , 70 стр . 1']"
4,Средства аэродромно-технического обеспечения полетов авиации,[],[]
5,(OlgaS) Sidorova,"[NAME(FIRST_NAME):(OlgaS), NAME(LAST_NAME):Sidorova]",[[NAME] '(OlgaS) Sidorova']
6,"""Разложение функций в ряд Фурье"" - автор А.Н. Колмогоров.","[NAME(LAST_NAME):Фурье"", NAME(LAST_NAME):Колмогоров.]","[[NAME] 'Фурье""', [NAME] 'Колмогоров.']"
7,Мосмарт,[],[]
8,POX,[],[]
9,f071:10c9:b5d1:182e:ce8b:71b:3883:4a99,[],[]


In [51]:
print("Конвертируем scanpatch train...")
db_sp_train = scanpatch_to_docbin(sp_train)
print("Конвертируем scanpatch test...")
db_sp_test  = scanpatch_to_docbin(sp_test)
print("Конвертируем AlexKly...")
db_alex     = alexkly_to_docbin(alex_df)

# Объединить все и рандомно разбить 90/10
all_docs = []
for db in [db_sp_train, db_sp_test, db_alex]:
    all_docs.extend(db.get_docs(nlp.vocab))

random.seed(42)
random.shuffle(all_docs)
split = int(len(all_docs) * 0.9)

train_db = DocBin(docs=all_docs[:split])
dev_db   = DocBin(docs=all_docs[split:])

train_db.to_disk('./data_spacy/train.spacy')
dev_db.to_disk('./data_spacy/dev.spacy')

train_docs = list(train_db.get_docs(nlp.vocab))
dev_docs   = list(dev_db.get_docs(nlp.vocab))
print(f"\nTrain: {len(train_docs)} docs, Dev: {len(dev_docs)} docs")

Конвертируем scanpatch train...
Конвертируем scanpatch test...
Конвертируем AlexKly...

Train: 9002 docs, Dev: 1001 docs


In [52]:
# 1. Статистика
for split_name, docs in [('train', train_docs), ('dev', dev_docs)]:
    label_counts = Counter(e.label_ for doc in docs for e in doc.ents)
    n_with_ents  = sum(1 for doc in docs if doc.ents)
    print(f"=== {split_name} ({len(docs)} docs) ===")
    print(f"  С сущностями: {n_with_ents} ({100*n_with_ents/len(docs):.1f}%)")
    for lbl, cnt in label_counts.most_common():
        print(f"  {lbl}: {cnt}")
    print()

# 2. Длины спанов
for label in ('NAME', 'ADDRESS'):
    lengths = [len(e.text) for doc in train_docs for e in doc.ents if e.label_ == label]
    if lengths:
        print(f"{label}: min={min(lengths)}, median={sorted(lengths)[len(lengths)//2]}, max={max(lengths)}")
print()

# 3. Топ-5 самых длинных спанов
for label in ('NAME', 'ADDRESS'):
    spans = sorted(
        [e.text for doc in train_docs for e in doc.ents if e.label_ == label],
        key=len, reverse=True
    )
    print(f"Топ-5 длинных [{label}]:")
    for s in spans[:5]:
        print(f"  {s!r}")
    print()

# 4. Случайные примеры с выделением в контексте
print("=== 20 случайных примеров ===")
docs_with_ents = [doc for doc in train_docs if doc.ents]
for doc in random.sample(docs_with_ents, min(20, len(docs_with_ents))):
    highlighted = doc.text
    for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
        highlighted = (
            highlighted[:ent.start_char]
            + f">>{ent.label_}:{highlighted[ent.start_char:ent.end_char]}<<"
            + highlighted[ent.end_char:]
        )
    print(highlighted[:300])
    print()

=== train (9002 docs) ===
  С сущностями: 2548 (28.3%)
  NAME: 3658
  ADDRESS: 2766

=== dev (1001 docs) ===
  С сущностями: 282 (28.2%)
  NAME: 393
  ADDRESS: 306

NAME: min=2, median=13, max=38
ADDRESS: min=2, median=10, max=92

Топ-5 длинных [NAME]:
  'профессора Николая Петровича Сидоренко'
  'Станиславом Григорьевичем Пасынковым'
  'Мирон Аркадьевич Малкиель-Жирмунский'
  'ВЛАДИ́МИР АЛЕКСА́НДРОВИЧ ЗЕЛЕ́НСКИЙ'
  'Станиславом Игоревичем Лаврентьевым'

Топ-5 длинных [ADDRESS]:
  'старом районе на перекрестке между проспектом Победы и улочкой Зеленого Мая в одном из домов'
  'головном офисе в Санкт-Петербурге (проспект Науки, 14, координаты: 60.010000,30.370000)'
  'восточной стороне города, у самого спуска к морю, расположена улица Приморская, дом 52'
  'Ростов-на-Дону, финансовый отдел (ул. Большая Садовая, 100, 47.230000,39.710000)'
  'Москва, отдел прототипирования (ул. Тверская, 15, коорд.: 55.757778,37.615556)'

=== 20 случайных примеров ===
﻿Файл с документом, который хранился 